# NVIDIA Nemotron Reasoning Challenge - E2E Pipeline

This notebook runs the full SFT + GRPO pipeline on Kaggle:
1. Load data (stratified by question type)
2. SFT warmup on Nemotron-3-Nano-30B with LoRA
3. GRPO reinforcement learning with correctness reward
4. Package submission.zip

**GPU**: RTX PRO 6000 (102 GB VRAM)

## 0. Setup

In [1]:
import os
import shutil
import sys

# === Install offline packages (datasets, trl, nemo) ===
# Use --no-deps: transitive deps (filelock, etc.) are already in the Kaggle base env
os.system("pip install -q --no-index --no-deps --find-links /kaggle/input/nemo-offline-packages datasets trl nemo")

# === Patch mamba_ssm to handle missing cutlass ===
# The utility script dir is READ-ONLY, so copy to a writable location first
UTIL_DIR = "/kaggle/usr/lib/nvidia-utility-script"
PATCHED_DIR = "/tmp/patched_util"

if os.path.exists(UTIL_DIR):
    if not os.path.exists(PATCHED_DIR):
        shutil.copytree(UTIL_DIR, PATCHED_DIR)

    init_path = os.path.join(PATCHED_DIR, "mamba_ssm", "__init__.py")
    if os.path.exists(init_path):
        with open(init_path) as f:
            text = f.read()
        if "from mamba_ssm.modules.mamba3 import Mamba3" in text:
            with open(init_path, "w") as f:
                f.write(text.replace(
                    "from mamba_ssm.modules.mamba3 import Mamba3",
                    "try:\n    from mamba_ssm.modules.mamba3 import Mamba3\n"
                    "except (ImportError, ModuleNotFoundError):\n    Mamba3 = None"
                ))
            print("Patched mamba_ssm to handle missing cutlass")

    if PATCHED_DIR not in sys.path:
        sys.path.insert(0, PATCHED_DIR)
    print(f"Utility script ready at {PATCHED_DIR}")
else:
    print(f"WARNING: Utility script not found at {UTIL_DIR}")


In [ ]:
import gc
import os
import re
import shutil
import stat
import sys
import zipfile

import kagglehub
import mamba_ssm
import pandas as pd
import torch
import torch.nn.functional as F
import trl.import_utils as _trl_iu
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

from data import (
    build_grpo_dataset,
    build_sft_dataset,
    classify_type,
    load_train_csv,
    stratified_sample,
)
from eval_utils import correctness_reward
from prompts import SYSTEM_PROMPT

# Force all trl optional-dep checks to False so it never tries to import them
for _attr in dir(_trl_iu):
    if _attr.startswith('is_') and _attr.endswith('_available') and callable(getattr(_trl_iu, _attr)):
        setattr(_trl_iu, _attr, lambda: False)

from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer  # noqa: E402, I001

# --- Pure PyTorch rmsnorm fallback (replaces triton kernel) ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# --- Triton ptxas fix ---
src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
dst = '/tmp/ptxas-blackwell'
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend  # noqa: E402
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), 'bin')
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = dst

    import triton.backends.nvidia.compiler as nv_compiler  # noqa: E402
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

print(f'PyTorch: {torch.__version__}')
print(f'mamba_ssm loaded from: {mamba_ssm.__file__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 1. Configuration

In [ ]:
# === CONFIGURATION ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
TEST_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv"
OUTPUT_DIR = "/kaggle/working/lora_adapter"
SUBMISSION_PATH = "/kaggle/working/submission.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LoRA
LORA_RANK = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# SFT warmup
SFT_SAMPLES_PER_TYPE = 5      # E2E test: 6 types * 5 = 30 samples
SFT_MAX_SEQ_LEN = 1024
SFT_LR = 2e-4

# GRPO reinforcement learning
GRPO_SAMPLES_PER_TYPE = 5     # E2E test: 6 types * 5 = 30 prompts
GRPO_MAX_COMPLETION = 512
GRPO_NUM_GENERATIONS = 2
GRPO_LR = 5e-6

# Shared
BATCH_SIZE = 1
GRAD_ACCUM = 4

print(f"Model path: {MODEL_PATH}")


## 2. Data Loading & Preparation

In [ ]:
train_df = load_train_csv(TRAIN_CSV)
print(f"Total training samples: {len(train_df)}")

# Classify and stratified sample
train_df['qtype'] = train_df['prompt'].apply(classify_type)
print(f"Types: {train_df['qtype'].value_counts().to_dict()}")

sft_df = stratified_sample(train_df, SFT_SAMPLES_PER_TYPE, seed=42)
grpo_df = stratified_sample(train_df, GRPO_SAMPLES_PER_TYPE, seed=123)
print(f"SFT samples: {len(sft_df)}, GRPO prompts: {len(grpo_df)}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Build datasets
sft_dataset = build_sft_dataset(sft_df.drop(columns=['qtype']), tokenizer, max_seq_length=SFT_MAX_SEQ_LEN)
print(f"\nSFT dataset: {len(sft_dataset)} samples")
print(f"SFT example (first 500 chars):\n{sft_dataset[0]['text'][:500]}")

grpo_dataset = build_grpo_dataset(grpo_df)
print(f"\nGRPO dataset: {len(grpo_dataset)} prompts")


## 3. Load Model & Tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f"Model loaded. Vocab: {len(tokenizer)}")

# --- Patches (must be done after every model load) ---
for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: slow path")

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn


## 4. Apply LoRA

In [ ]:
peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 5. Train (SFT + GRPO)


In [ ]:
from transformers import TrainerCallback

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"[Step {state.global_step}] {logs}", flush=True)

gc.collect()
torch.cuda.empty_cache()

# === Stage 1: SFT Warmup ===
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=SFT_LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    optim="adamw_torch",
    max_grad_norm=1.0,
    max_length=SFT_MAX_SEQ_LEN,
    packing=False,
    seed=42,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    callbacks=[PrintLossCallback()],
)

print("Starting SFT warmup...")
sft_trainer.train()

# Save SFT adapter as fallback
model.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to {OUTPUT_DIR} (fallback)")

# Cleanup SFT trainer, keep model for GRPO
del sft_trainer
gc.collect()
torch.cuda.empty_cache()
print("SFT cleanup done.")

# === Stage 2: GRPO (disabled for baseline) ===
print("GRPO disabled for baseline run. Using SFT-only adapter.")


## 6. Save & Package Submission


In [ ]:
# Save final GRPO adapter (overwrites SFT fallback)
model.save_pretrained(OUTPUT_DIR)
print(f"Final adapter saved to {OUTPUT_DIR}")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size / 1024:.1f} KB)")

# Package submission.zip
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

size_mb = os.path.getsize(SUBMISSION_PATH) / (1024 * 1024)
print(f"\nSubmission: {SUBMISSION_PATH} ({size_mb:.1f} MB)")

with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("submission.zip ready!")


## 7. Quick Sanity Check

Run inference on a few examples to verify the adapter works.


In [ ]:
model.eval()
test_df = pd.read_csv(TEST_CSV)
sample = test_df.head(3)

for _, row in sample.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    boxed = re.findall(r"\\boxed\{([^}]+)\}", response)
    answer = boxed[-1] if boxed else "N/A"

    print(f"ID: {row['id']}")
    print(f"Response (last 200 chars): ...{response[-200:]}")
    print(f"Extracted answer: {answer}")
    print("-" * 40)


In [ ]:
print("Done! Submit submission.zip to the competition.")
print(f"File: {SUBMISSION_PATH}")
